# 06 — Per-architecture metric collection

This notebook confirms how the comparison stage represents one trial per architecture and how per-architecture test metrics are assembled into a single comparable structure. Since the data mount and trained checkpoints are unavailable, we populate the real `TrialRecord` dataclass with mock metrics that mimic the headline metric keys the pipeline reads.

Modules exercised: `pipelines/benchmark_pipeline/results.py` (`TrialRecord`, `_HEADLINE_METRICS`).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import warnings
warnings.filterwarnings("ignore")

import numpy             as np
import torch
import matplotlib.pyplot as plt

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(False)

DEVICE = torch.device("cpu")

plt.rcParams.update({
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
    "axes.axisbelow" : True,
})

print("bootstrap ready  —  torch", torch.__version__, " device", DEVICE)


## The headline metric contract

The comparison report ranks models on a fixed list of headline metric keys, each with a human label. We print that list so the mock metrics below use the real keys.

In [ ]:
from pipelines.benchmark_pipeline.results import _HEADLINE_METRICS

for key, label in _HEADLINE_METRICS:
    print(f"{key:32s} -> {label}")

## Building mock trial records

Each architecture becomes one `TrialRecord` carrying its parameter count, size-match summary, and a metrics dict keyed by the headline metric names. The mock values are seeded so the notebook is reproducible. In a real run these come from `inference/<run>/metrics.json`.

In [ ]:
from pathlib import Path
from pipelines.benchmark_pipeline.results import TrialRecord

models = ["unet", "resunet", "attention_unet", "segformer", "fpn", "dense_unet"]
rng    = np.random.default_rng(SEED)

HEADLINE_KEYS = [k for k, _ in _HEADLINE_METRICS]
HIGHER_KEYS   = {"overall_r2_gt", "psnr_db_gt", "pixel_r2_gt_mean", "pixel_cosine_gt_mean", "ssim_gt_elev_mean"}

def mock_metric(key, skill):
    if key in HIGHER_KEYS:
        return float(0.5 + 0.45 * skill + 0.02 * rng.standard_normal())
    return float(1.0 - 0.7 * skill + 0.02 * rng.standard_normal())

records = []
for i, name in enumerate(models):
    skill  = 1.0 - i / (len(models) - 1)
    record = TrialRecord(name=name, run_dir=Path("/tmp") / name)
    record.parameters = 300_000 + int(2_000 * rng.standard_normal())
    record.size_match = {"scale": round(0.3 + 0.05 * i, 3), "deviation_pct": round(0.3 * rng.standard_normal(), 3), "overrides": {"features": [16, 32]}}
    record.metrics    = {key: mock_metric(key, skill) for key in HEADLINE_KEYS}
    records.append(record)

for r in records:
    print(f"{r.name:16s} params={r.parameters:>8,}  R2={r.metrics['overall_r2_gt']:.3f}  RMSE={r.metrics['curve_rmse_gt']:.3f}")

## Assembling the metric matrix

The per-record metric dicts are collated into one matrix with architectures as rows and headline metrics as columns. This is the structure the ranking logic consumes.

In [ ]:
labels = [label for _, label in _HEADLINE_METRICS]
matrix = np.array([[r.metrics[key] for key, _ in _HEADLINE_METRICS] for r in records])

header = "model".ljust(16) + "".join(f"{lab:>11s}" for lab in labels)
print(header)
print("-" * len(header))
for r, row in zip(records, matrix):
    print(r.name.ljust(16) + "".join(f"{v:>11.3f}" for v in row))

## Visual confirmation

We column-normalise the metric matrix (per metric, best = 1) and display it as a heatmap. Because the mock records were built with monotonically decreasing skill, the brighter cells should cluster toward the top rows for higher-is-better metrics and require reading the printed table for lower-is-better metrics; the normalisation here orients all columns so brighter is better.

In [ ]:
norm = np.zeros_like(matrix)
for j, (key, _) in enumerate(_HEADLINE_METRICS):
    col   = matrix[:, j]
    good  = col if key in HIGHER_KEYS else -col
    lo, hi = good.min(), good.max()
    norm[:, j] = (good - lo) / (hi - lo + 1e-12)

fig, ax = plt.subplots(figsize=(9, 4.5))
im = ax.imshow(norm, cmap="viridis", aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=30, ha="right")
ax.set_yticks(range(len(records)))
ax.set_yticklabels([r.name for r in records])
ax.set_title("Per-architecture metrics (column-normalised, brighter = better)")
ax.grid(False)
fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
fig.tight_layout()
plt.show()

## Expected visual outcome

The printed metric matrix has one row per architecture and one column per headline metric. After per-column orientation the heatmap is brightest in the upper rows, matching the decreasing skill the mock data encodes. These records feed directly into the ranking notebook.